In [ ]:
import pandas as pd

In [ ]:
data_eda = pd.read_csv('../data/index_data.csv')

In [ ]:
for i in data_eda["conm"].unique():
    print(i)

In [ ]:
# tic_list = ["MET.I3", "OIG.I", "ENE.I", "MET.I","I0003", "I0048", "I0049", "I0050", "I0051", "I0052", "I0053", "I0054", "I0055", "I0056", "I0057", "I0058", "I0059", "I0060", "I0061", "I0062", "I0063", "I0064", "I0065", "I0066"]

In [ ]:
tic_list = ["I0003", "I0048", "I0049", "I0050", "I0051", "I0052", "I0053", "I0054", "I0055", "I0056", "I0057", "I0058", "I0059", "I0060", "I0061", "I0062", "I0063", "I0064", "I0065", "I0066"]

## 데이터 설명
- tic : 거래심볼
- datadate : 데이터 날짜
- gvkeyx : global index key
- conm : Index name
- dvpsxd : 지수 구성 종목의 보통주에 대한 배당금의 총액
- newnum : 현재 거래일의 지수 구성 요소수
- oldnum : 이전 거래일의 지수 구성 항목수 
- prccd : 해당 증권의 당일 종가, 거래량이 없는 매수 매도 가격의 경우 종가가 거래량이 포함된 마지막 거래 종가를 나타냄, 매수 매도 가격은 최고/최저 범위 밖에서 종가 형성 가능
- prccddiv : 지수의 일일 종가 수익률 가격 상승과 일일 배당금 재투자, 재투자된 배당금에 지급된 배당금의 복리효과 반영함.
- prccddivn : 지수의 일일 종가 수익률로 가격 상승과 세금을 제한후 순 배당을 재투자한 경우를 가정함.
- prchd : 거래일의 최고 거래 가격을 나타냄
- PRCLD : 매수 / 매도 책정의 경우 거래일의 가장 낮은 거래 가격을 나타냄. 장 마감전 마지막 입찰가를 나타냄. 매수 매도 가격 차이가 최신 매도 가격의 50%보다 큰 경우 이전 입찰 가격을 타나냄

In [ ]:
using_data = data_eda[data_eda["tic"].isin(tic_list)]

In [ ]:
using_data = using_data.sort_values(by=["datadate"])

In [ ]:
using_data.reset_index(drop=True, inplace=True)

In [ ]:
unique_pairs = using_data[['tic', 'conm']].drop_duplicates().reset_index(drop=True)

In [ ]:
mapping_dict = unique_pairs.set_index('tic')['conm'].to_dict()

In [ ]:
tic_mapping = {'I0003': 'SP500',
 'MET.I3': 'Gold',
 'ENE.I': 'Energy',
 'MET.I': 'Metals_Mining',
 'OIG.I': 'Oil_Gas_Consumable_Fuels',
 'I0063': 'Sweden',
 'I0059': 'Netherlnd',
 'I0049': 'Austria',
 'I0058': 'Mexico',
 'I0057': 'Malaysia',
 'I0062': 'Spain',
 'I0061': 'Singapore',
 'I0054': 'Hong Kong',
 'I0048': 'Australia',
 'I0051': 'Canada',
 'I0056': 'Japan',
 'I0055': 'Italy',
 'I0053': 'Germany',
 'I0052': 'France',
 'I0064': 'Switzrlnd',
 'I0066': 'Belgium',
 'I0065': 'Utd_Kgdm'}

In [ ]:
using_data["tic_map"] = using_data.tic.map(tic_mapping, na_action='ignore')

In [ ]:
data_start = "1999-01-04"

In [ ]:
index_data = using_data[using_data["datadate"] >=data_start]

In [ ]:
index_data.reset_index(drop=True, inplace=True)

In [ ]:
# 전체 티커 목록
all_tickers = index_data["tic_map"].unique()

# 각 티커별 보유 날짜 집합
date_sets = {
    ticker: set(index_data[index_data["tic_map"] == ticker]["datadate"])
    for ticker in all_tickers
}

# 모든 티커가 공통으로 가진 날짜 (교집합)
common_dates = set.intersection(*date_sets.values())

# 누락된 날짜 확인: 전체 날짜에서 공통 날짜 빼기
all_dates = set(index_data["datadate"].unique())
missing_dates = sorted(all_dates - common_dates)

print("✅ 삭제 대상 날짜 목록:")
print(missing_dates)


In [ ]:
# 모든 티커가 존재하는 날짜 교집합
valid_dates = set.intersection(*[
    set(index_data[index_data["tic_map"] == i]["datadate"]) 
    for i in index_data.tic_map.unique()
])

# 그 날짜만 필터링
index_data = index_data[index_data["datadate"].isin(valid_dates)]


In [ ]:
subset = index_data[index_data.tic_map == "SP500"].copy()


In [ ]:
tic_mapping.values()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# for i in tic_mapping.values():
#     subset = index_data[index_data.tic_map == i].copy()

#     print("High price 이상치 상위 5개:")
#     print(subset[['datadate', 'tic_map', 'prchd']].sort_values('prchd', ascending=False).head(5))

#     print("\nClose price 이상치 상위 5개:")
#     print(subset[['datadate', 'tic_map', 'prccd']].sort_values('prccd', ascending=False).head(5))

#     print("\nLow price 이상치 상위 5개:")
#     print(subset[['datadate', 'tic_map', 'prcld']].sort_values('prcld', ascending=False).head(5))


In [ ]:
def find_all_price_jumps(df, price_col='prccd', threshold=2.0):
    jump_all = []

    for name, group in df.groupby('tic_map'):
        group = group.sort_values('datadate').copy()
        group['datadate'] = pd.to_datetime(group['datadate'])
        group['prev_price'] = group[price_col].shift(1)
        group['ratio'] = group[price_col] / group['prev_price']

        # 변화 비율이 너무 큰 경우 (상승 or 하락)
        jump_mask = (group['ratio'] > threshold) | (group['ratio'] < 1 / threshold)
        jumps = group.loc[jump_mask, ['tic_map', 'datadate', 'prccd', 'prev_price', 'ratio']]
        jump_all.append(jumps)

    return pd.concat(jump_all).sort_values(['tic_map', 'datadate']).reset_index(drop=True)


In [ ]:
jumps_all = find_all_price_jumps(index_data, price_col='prccd', threshold=2.0)


In [ ]:
index_data["tic_map"].unique()

In [ ]:
jumps_all

In [ ]:
subset = index_data[index_data["tic_map"]=="Belgium"]
subset = subset.set_index('datadate').sort_index()

In [ ]:
import numpy as np
import pandas as pd

# 1. 저가 이상치 정제 함수
def clean_low_price(df, price_col='prcld', ref_col='prccd',
                    low_thresh_ratio=0.5, replace_with='ref'):
    df = df.copy()
    mask = df[price_col] < df[ref_col] * low_thresh_ratio
    if replace_with == 'nan':
        df.loc[mask, price_col] = np.nan
    elif replace_with == 'ref':
        df.loc[mask, price_col] = df.loc[mask, ref_col]
    return df

# 2. 고가 이상치 정제 함수
def clean_high_price(df, price_col='prchd', ref_col='prccd',
                     threshold_ratio=1.5, replace_with='ref'):
    df = df.copy()
    mask = df[price_col] > df[ref_col] * threshold_ratio
    if replace_with == 'nan':
        df.loc[mask, price_col] = np.nan
    elif replace_with == 'ref':
        df.loc[mask, price_col] = df.loc[mask, ref_col]
    return df

# 3. Italy 수동 이상치 + 저가 정제
def clean_italy(df):
    df = df.copy()
    italy = df[df['tic_map'] == 'Italy'].copy()
    italy['datadate'] = pd.to_datetime(italy['datadate'])

    # 수동 보정 (2000-06-30)
    mask = italy['datadate'] == '2000-06-30'
    before = italy.loc[italy['datadate'] == '2000-06-29', 'prccd'].values[0]
    after = italy.loc[italy['datadate'] == '2000-07-03', 'prccd'].values[0]
    corrected = (before + after) / 2
    italy.loc[mask, 'prccd'] = corrected
    italy.loc[mask, 'prchd'] = corrected

    # 저가 정제
    italy = clean_low_price(italy, price_col='prcld', ref_col='prccd',
                            low_thresh_ratio=0.5, replace_with='ref')
    return italy

# 4. 리버스 스플릿 스케일 적용 함수
def apply_reverse_split_scaling(df, cutoff='2016-11-07', scale=1.0):
    df = df.copy()
    df['datadate'] = pd.to_datetime(df['datadate'])

    before = df[df['datadate'] < cutoff].copy()
    after = df[df['datadate'] >= cutoff].copy()

    if not before.empty:
        before[['prccd', 'prchd', 'prcld']] *= scale

    return pd.concat([before, after]).sort_values('datadate').reset_index(drop=True)

# 5. 전체 정제 함수 (이탈리아, 국가별 스케일 포함)
def clean_all_index_data(df):
    df = df.copy()
    df['datadate'] = pd.to_datetime(df['datadate'])

    cleaned_all = []

    # ✅ 국가별 리버스 스플릿 스케일링 비율
    split_scales = {
        'Japan': 4.0,
        'Malaysia': 4.0,
        'Singapore': 2.0,
        'Utd_Kgdm': 2.0
    }

    for name, group in df.groupby('tic_map'):
        group = group.sort_values('datadate').copy()

        if name == 'Italy':
            cleaned = clean_italy(group)

        elif name in split_scales:
            group = apply_reverse_split_scaling(group, cutoff='2016-11-07', scale=split_scales[name])
            group = clean_low_price(group, price_col='prcld', ref_col='prccd',
                                    low_thresh_ratio=0.5, replace_with='ref')
            group = clean_high_price(group, price_col='prchd', ref_col='prccd',
                                     threshold_ratio=1.5, replace_with='ref')
            cleaned = group

        else:
            group = clean_low_price(group, price_col='prcld', ref_col='prccd',
                                    low_thresh_ratio=0.5, replace_with='ref')
            group = clean_high_price(group, price_col='prchd', ref_col='prccd',
                                     threshold_ratio=1.5, replace_with='ref')
            cleaned = group

        cleaned_all.append(cleaned)

    return pd.concat(cleaned_all).reset_index(drop=True)
index_data_cleaned = clean_all_index_data(index_data)


In [ ]:
for i in tic_mapping.values():
    print(i)

    subset = index_data_cleaned[index_data_cleaned.tic_map == i].copy()
    subset = subset.sort_values('datadate')

    # 20일마다 datadate 추출 (인덱스 기반 또는 실제 날짜 기준)
    tick_idx = subset.index[::90]  # 20개마다 한 번
    tick_labels = subset['datadate'].iloc[::90]

    plt.figure(figsize=(12, 6))
    plt.plot(subset['datadate'], subset['prccd'], label='Close Price (prccd)')
    plt.plot(subset['datadate'], subset['prchd'], label='High Price (prchd)')
    plt.plot(subset['datadate'], subset['prcld'], label='Low Price (prcld)')

    plt.title(f'Price Series (Close / High / Low) - {i}')
    plt.xlabel('Date')
    plt.ylabel('Price')
    plt.xticks(tick_labels, rotation=45)  # x축 눈금 설정
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
index_data_cleaned

In [ ]:
index_data_cleaned.sort_values(by=["datadate", "tic"], inplace=True)

In [ ]:
index_data_cleaned.reset_index(drop=True, inplace=True)

In [ ]:
index_data_cleaned.to_csv("../data/select_index_data_cleaned_v1.csv", index=False)

In [ ]:
import pandas as pd

In [ ]:
imp_index = pd.read_csv("../data/select_index_data_cleaned_v1.csv")

In [ ]:
imp_index["tic_map"].unique()

In [ ]:
import yfinance  as yf
import pandas as pd


In [ ]:
tickers = [
    # 금속
    "GC=F", "SI=F",  "HG=F",
    # 에너지
    "CL=F", "NG=F", "HO=F",
    # 농산물
    "ZC=F", "ZL=F", 
    # 축산물
    # "HE=F","PL=F", "ZO=F",
    # 기타 원자재
    "KC=F", "SB=F", "CC=F", "CT=F"]


In [ ]:
ticker_name_map = {
    # 금속
    "GC=F": "Gold",
    "SI=F": "Silver",
    "PL=F": "Platinum",
    "HG=F": "Copper",
    
    # 에너지
    "CL=F": "Crude Oil",
    "NG=F": "Natural Gas",
    "HO=F": "Heating Oil",
    
    # 농산물
    "ZC=F": "Corn",
    "ZL=F": "Soybean Oil",
    "ZO=F": "Oats",
    
    # 축산물
    # "HE=F": "Lean Hogs",
    
    # 기타 원자재
    "KC=F": "Coffee",
    "SB=F": "Sugar",
    "CC=F": "Cocoa",
    "CT=F": "Cotton"
}


In [ ]:

# 수집 기간
start_date = data_start
end_date = "2025-12-31"

# 결과 저장
futures_data = []

# 티커별 데이터 다운로드 및 정리
for symbol in tickers:
    print(f"Downloading: {symbol}")
    df = yf.download(symbol, start=start_date, end=end_date)
    df["Symbol"] = symbol
    df["Date"] = df.index
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    futures_data.append(df)

# 하나의 DataFrame으로 통합
futures_df = pd.concat(futures_data, ignore_index=True)

# Symbol과 Date 기준으로 정리
futures_df = futures_df[["Symbol", "Date", "Open", "High", "Low", "Close", "Volume"]]


In [ ]:
futures_df.columns

In [ ]:
futures_df["tic_map"] = futures_df.Symbol.map(ticker_name_map, na_action='ignore')

In [ ]:
# for i in futures_df["Symbol"].unique():
#     print(i)
#     imp = futures_df[futures_df["Symbol"]==i].copy()
#     print(imp.Date.min())
#     print(imp.Date.max())

In [ ]:
futures_df["Date"] = futures_df["Date"].dt.strftime("%Y-%m-%d")


In [ ]:
for i in futures_df["Symbol"].unique():
    print(i)
    imp = futures_df[futures_df["Symbol"]==i].copy()
    print(imp.Date.min())

In [ ]:
slice_future = futures_df[futures_df["Date"] >= "2000-09-01"]

In [ ]:
slice_future.tic_map.unique()

In [ ]:
slice_future.reset_index(drop=True, inplace=True)

In [ ]:
import matplotlib.pyplot as plt


In [ ]:
for i in slice_future.Symbol.unique():
    print(i)

    subset = slice_future[slice_future["Symbol"] == i].copy()
    subset = subset.sort_values('Date')

    # 20일마다 datadate 추출 (인덱스 기반 또는 실제 날짜 기준)
    tick_idx = subset.index[::90]  # 20개마다 한 번
    tick_labels = subset['Date'].iloc[::90]

    plt.figure(figsize=(12, 6))
    plt.plot(subset['Date'], subset['Close'], label='Close Price (prccd)')
    # plt.plot(subset['Date'], subset['High'], label='High Price (prchd)')
    # plt.plot(subset['Date'], subset['Low'], label='Low Price (prcld)')
    # plt.plot(subset['Date'], subset['Open'], label='Open Price (prchd)')


    plt.title(f'Price Series (Close / High / Low) - {i}')
    plt.xlabel('Date')
    plt.ylabel('Price')
    plt.xticks(tick_labels, rotation=45)  # x축 눈금 설정
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
hg_df = slice_future[slice_future["Symbol"] == "CL=F"]
print(hg_df[hg_df["Close"] < 0])  # 음수인 행 필터링

In [ ]:
drop_date = ['2001-09-11',
 '2001-09-12',
 '2001-09-13',
 '2001-09-14',
 '2002-02-18',
 '2002-07-04',
 '2004-06-11',
 '2004-11-25',
 '2005-11-24',
 '2006-09-04',
 '2006-09-28',
 '2006-11-23',
 '2006-12-25',
 '2007-01-01',
 '2007-01-02',
 '2012-10-19',
 '2023-11-23']

In [ ]:
slice_future = slice_future[slice_future["Date"] <= "2024-12-31"]

In [ ]:
slice_future = slice_future[~slice_future['Date'].isin(drop_date)]  # isin: 포함 여부, ~: 부정 (즉 제거)


In [ ]:
imp_index = imp_index[(imp_index["datadate"] <= "2024-12-31") & (imp_index["datadate"] >= "2000-09-01")]

In [ ]:
index_drop_date = ['2016-10-10', '2016-11-11']

In [ ]:
imp_index = imp_index[~imp_index['datadate'].isin(index_drop_date)]  # isin: 포함 여부, ~: 부정 (즉 제거)


In [ ]:
for tic in slice_future.tic_map.unique():
    imp = slice_future[slice_future["tic_map"] == tic]
    start_date = imp["Date"].min()
    end_date = imp["Date"].max()
    print(f"{tic:15} | {imp.shape[0]} rows | {start_date} ~ {end_date}")


In [ ]:
# 자산별 날짜 set 저장
date_dict = {}
for tic in slice_future.tic_map.unique():
    imp = slice_future[slice_future["tic_map"] == tic]
    date_dict[tic] = set(imp["Date"])

# 1. 전체 자산의 날짜 교집합 (모두 있는 날짜)
common_dates = set.intersection(*date_dict.values())
print(f"✔ 모든 자산에 공통된 날짜 수: {len(common_dates)}")

# # 2. 전체 자산의 날짜 합집합 (모든 날짜들)
# all_dates = set.union(*date_dict.values())
# print(f"📅 전체 자산 중 하나라도 포함된 날짜 수: {len(all_dates)}")

# # 3. 자산별로 누락된 날짜 확인
# for tic, dates in date_dict.items():
#     missing = sorted(all_dates - dates)
#     print(f"{tic:15} | 누락된 날짜 수: {len(missing)}")
#     if len(missing) > 0:
#         print(f"  ⤷ 예시: {missing[:3]} ... {missing[-3:]}")


In [ ]:
common_dates = sorted(common_dates)

In [ ]:
slice_future = slice_future[slice_future['Date'].isin(common_dates)]  # isin: 포함 여부, ~: 부정 (즉 제거)


In [ ]:
imp_index = imp_index[imp_index['datadate'].isin(common_dates)]  # isin: 포함 여부, ~: 부정 (즉 제거)


In [ ]:
slice_future.reset_index(drop=True, inplace=True)

In [ ]:
slice_future

In [ ]:
for i in slice_future.Symbol.unique():
    print(i)

    subset = slice_future[slice_future["Symbol"] == i].copy()
    subset = subset.sort_values('Date')

    # 20일마다 datadate 추출 (인덱스 기반 또는 실제 날짜 기준)
    tick_idx = subset.index[::90]  # 20개마다 한 번
    tick_labels = subset['Date'].iloc[::90]

    plt.figure(figsize=(12, 6))
    plt.plot(subset['Date'], subset['Close'], label='Close Price (prccd)')
    # plt.plot(subset['Date'], subset['High'], label='High Price (prchd)')
    # plt.plot(subset['Date'], subset['Low'], label='Low Price (prcld)')
    # plt.plot(subset['Date'], subset['Open'], label='Open Price (prchd)')


    plt.title(f'Price Series (Close / High / Low) - {i}')
    plt.xlabel('Date')
    plt.ylabel('Price')
    plt.xticks(tick_labels, rotation=45)  # x축 눈금 설정
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
rp_weights

In [ ]:
import numpy as np

In [ ]:
hg_df = slice_future[slice_future["Symbol"] == "ZR=F"]


In [ ]:
hg_df = slice_future[slice_future["Symbol"] == "ZR=F"]
print(hg_df[hg_df["Close"] < 20])  # 음수인 행 필터링

In [ ]:
slice_future

In [ ]:
slice_future.set_index("Date", inplace=True)

In [ ]:
slice_future.to_csv("../data/futures_data_v1.csv", index=True)

In [ ]:
slice_future["tic_map"].unique()

In [ ]:
for i in slice_future.tic_map.unique():
    print(i)
    imp = slice_future[slice_future["tic_map"]==i].copy()
    print(imp.index.min())

In [ ]:
imp_index.sort_values(by=["datadate", "tic"], inplace=True)

In [ ]:
imp_index.reset_index(drop=True, inplace=True)

In [ ]:
imp_index.to_csv("../data/select_index_data_cleaned_v2.csv", index=False)

In [ ]:
imp_index